# SIED-Thai Original — 5 Seeds × 100 Epochs (Frozen WangchanBERTa, Kaggle T4 ×2)

เวอร์ชันนี้ยึดจาก notebook ต้นฉบับ:
`wanchanberta-x-transformers-sied-optimzed(2).ipynb`

## Dataset / protocol
- SIED-Thai **original / unbalanced**
- Train = 1,728
- Validation = 192
- Test = 480
- Split seed = 42
- `MAX_LEN = 128`
- `BATCH_SIZE = 64`
- WangchanBERTa **frozen**
- X-Transformers decoder + Attention Pooling + classifier
- ใช้ Optuna best params เดิม
- ไม่รัน Optuna ซ้ำ
- 5 independent seeds: `[13, 21, 42, 87, 123]`
- **100 epochs ต่อ run**
- รวม repeated-run training budget = **500 decoder-training epochs**
- Threshold tuning เดิม: `0.10 ... 0.85`, step `0.05`
- Mean / Median / Sample Variance / Sample SD
- Precision / Recall / F1 / Support แยกครบ 6 อารมณ์
- ถ้า Kaggle ตรวจพบ **T4 ×2** จะใช้ `torch.nn.DataParallel` อัตโนมัติ

## Fixed best params จาก Optuna เดิม
```python
{
    "depth": 2,
    "heads": 4,
    "attn_dropout": 0.42780092819742693,
    "ff_dropout": 0.45659903330825574,
    "ff_mult": 3,
    "lr": 4.4638200794376354e-05,
    "weight_decay": 0.00014211620758672024
}
```

> WangchanBERTa encode/cache เพียงครั้งเดียว แล้วฝึก decoder/pooling/classifier ใหม่ตั้งแต่ต้นในแต่ละ seed.


In [7]:
!pip install -q "transformers==4.44.2" "tokenizers==0.19.1" sentencepiece x-transformers scikit-learn pandas tqdm

In [8]:
import os
import glob
import gc
import random
import time
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from transformers import (
    AutoTokenizer,
    AutoModel,
    get_linear_schedule_with_warmup
)

from x_transformers import Decoder

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    f1_score,
    accuracy_score,
    precision_score,
    recall_score,
    hamming_loss
)

PRIMARY_DEVICE = torch.device(
    "cuda:0" if torch.cuda.is_available() else "cpu"
)

GPU_COUNT = torch.cuda.device_count()
USE_DATA_PARALLEL = GPU_COUNT >= 2

print("PRIMARY_DEVICE:", PRIMARY_DEVICE)
print("GPU COUNT:", GPU_COUNT)

for i in range(GPU_COUNT):
    print(f"GPU {i}:", torch.cuda.get_device_name(i))

print("USE_DATA_PARALLEL:", USE_DATA_PARALLEL)

if GPU_COUNT < 2:
    print("WARNING: ไม่พบ GPU 2 ใบ — โค้ดจะรันด้วย device ที่มีอยู่")


PRIMARY_DEVICE: cuda:0
GPU COUNT: 2
GPU 0: Tesla T4
GPU 1: Tesla T4
USE_DATA_PARALLEL: True


In [9]:
# =========================================================
# CONFIG
# =========================================================

MODEL_NAME = "airesearch/wangchanberta-base-att-spm-uncased"

MAX_LEN = 128
BATCH_SIZE = 64
CACHE_BATCH_SIZE = 32

EPOCHS = 100

# split เดิมจาก notebook ต้นฉบับ
SPLIT_SEED = 42

# 5 independent training runs
TRAIN_SEEDS = [
    13,
    21,
    42,
    87,
    123
]

TEXT_COLUMN = "Tweets"

LABEL_COLUMNS = [
    "Happiness",
    "Sadness",
    "Anger",
    "Disgust",
    "Surprise",
    "Fear"
]

NUM_LABELS = 6

# Best params จาก Optuna ใน notebook ต้นฉบับ
FIXED_BEST_PARAMS = {
    "depth": 2,
    "heads": 4,
    "attn_dropout": 0.42780092819742693,
    "ff_dropout": 0.45659903330825574,
    "ff_mult": 3,
    "lr": 4.4638200794376354e-05,
    "weight_decay": 0.00014211620758672024
}

OUTPUT_DIR = (
    "/kaggle/working"
    if os.path.exists("/kaggle/working")
    else "/content"
)

os.makedirs(
    OUTPUT_DIR,
    exist_ok=True
)

print("EPOCHS:", EPOCHS)
print("TRAIN SEEDS:", TRAIN_SEEDS)
print("FIXED BEST PARAMS:", FIXED_BEST_PARAMS)

EPOCHS: 100
TRAIN SEEDS: [13, 21, 42, 87, 123]
FIXED BEST PARAMS: {'depth': 2, 'heads': 4, 'attn_dropout': 0.42780092819742693, 'ff_dropout': 0.45659903330825574, 'ff_mult': 3, 'lr': 4.4638200794376354e-05, 'weight_decay': 0.00014211620758672024}


In [10]:
# =========================================================
# REPRODUCIBILITY
# =========================================================

def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)

    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


seed_everything(SPLIT_SEED)

In [11]:
# =========================================================
# FIND SIED-THAI ORIGINAL
# =========================================================

import os
import glob

def find_sied_csv():

    # path ที่เป็นไฟล์โดยตรง
    candidates = [
        "/content/SIED-Thai.csv",
        "/content/SIED-Thai(1).csv",
        "/mnt/data/SIED-Thai.csv",
        "/mnt/data/SIED-Thai(1).csv",
    ]

    for path in candidates:
        if os.path.isfile(path):
            return path

    # ค้นใน Kaggle ทุกระดับ
    hits = glob.glob(
        "/kaggle/input/**/*.csv",
        recursive=True
    )

    # เอาไฟล์ที่ชื่อเกี่ยวกับ SIED ก่อน
    sied_hits = [
        path for path in hits
        if "sied" in os.path.basename(path).lower()
    ]

    if sied_hits:
        print("พบไฟล์ SIED:")
        for path in sied_hits:
            print(" -", path)

        return sied_hits[0]

    # fallback ถ้า dataset นี้มี CSV แค่ไฟล์เดียว
    if len(hits) == 1:
        return hits[0]

    raise FileNotFoundError(
        "ไม่พบไฟล์ SIED-Thai CSV ใน /kaggle/input/"
    )


CSV_PATH = find_sied_csv()

print("CSV_PATH:", CSV_PATH)
print("IS FILE:", os.path.isfile(CSV_PATH))

พบไฟล์ SIED:
 - /kaggle/input/datasets/tkorg99/sied-thai/SIED-Thai.csv
CSV_PATH: /kaggle/input/datasets/tkorg99/sied-thai/SIED-Thai.csv
IS FILE: True


In [12]:
# =========================================================
# LOAD DATASET
# =========================================================

df = pd.read_csv(CSV_PATH)

required_columns = [
    TEXT_COLUMN
] + LABEL_COLUMNS

missing = [
    col for col in required_columns
    if col not in df.columns
]

if missing:
    raise ValueError(
        f"Missing columns: {missing}"
    )

# คงแถวทั้งหมดเหมือนต้นฉบับ
df = df.copy()

df[TEXT_COLUMN] = (
    df[TEXT_COLUMN]
    .astype(str)
)

for col in LABEL_COLUMNS:

    df[col] = pd.to_numeric(
        df[col],
        errors="coerce"
    ).fillna(0).astype(int)


print("TOTAL ROWS:", len(df))

print("\nLABEL DISTRIBUTION")
print(
    df[LABEL_COLUMNS]
    .sum()
)

TOTAL ROWS: 2400

LABEL DISTRIBUTION
Happiness     170
Sadness      1886
Anger         312
Disgust       467
Surprise      162
Fear          304
dtype: int64


In [13]:
# =========================================================
# FIXED SPLIT — EXACTLY LIKE ORIGINAL NOTEBOOK
# 80/20 first, then 90/10 within the 80%
# Final = 72% train / 8% val / 20% test
# =========================================================

train_df, test_df = train_test_split(
    df,
    test_size=0.2,
    random_state=SPLIT_SEED,
    shuffle=True
)

train_df, val_df = train_test_split(
    train_df,
    test_size=0.1,
    random_state=SPLIT_SEED,
    shuffle=True
)

train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

print("TRAIN:", len(train_df))
print("VAL  :", len(val_df))
print("TEST :", len(test_df))

assert len(train_df) == 1728, len(train_df)
assert len(val_df) == 192, len(val_df)
assert len(test_df) == 480, len(test_df)

for name, frame in [
    ("TRAIN", train_df),
    ("VAL", val_df),
    ("TEST", test_df)
]:
    print(f"\n{name} LABEL DISTRIBUTION")
    print(frame[LABEL_COLUMNS].sum())

TRAIN: 1728
VAL  : 192
TEST : 480

TRAIN LABEL DISTRIBUTION
Happiness     122
Sadness      1346
Anger         229
Disgust       339
Surprise      124
Fear          233
dtype: int64

VAL LABEL DISTRIBUTION
Happiness     13
Sadness      153
Anger         30
Disgust       43
Surprise      10
Fear          20
dtype: int64

TEST LABEL DISTRIBUTION
Happiness     35
Sadness      387
Anger         53
Disgust       85
Surprise      28
Fear          51
dtype: int64


In [14]:
# =========================================================
# POS WEIGHT — EXACT ORIGINAL FORMULA
# total_samples / positive_count
# =========================================================

label_counts = (
    train_df[LABEL_COLUMNS]
    .sum()
    .values
    .astype(np.float32)
)

total_samples = len(train_df)

pos_weights_np = (
    total_samples
    / (label_counts + 1e-6)
)

print("LABEL COUNTS:", label_counts)
print("POS WEIGHTS :", pos_weights_np)

# Expected approximately:
# [14.1639, 1.2838, 7.5459, 5.0973, 13.9355, 7.4163]

LABEL COUNTS: [ 122. 1346.  229.  339.  124.  233.]
POS WEIGHTS : [14.163935   1.2838038  7.5458517  5.0973454 13.935484   7.416309 ]


In [15]:
# =========================================================
# TOKENIZER + FROZEN WANGCHANBERTA
# =========================================================

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME
)

encoder = AutoModel.from_pretrained(
    MODEL_NAME
).to(PRIMARY_DEVICE)

for param in encoder.parameters():
    param.requires_grad = False

encoder.eval()

HIDDEN_SIZE = encoder.config.hidden_size

print("HIDDEN_SIZE:", HIDDEN_SIZE)

print(
    "TRAINABLE ENCODER PARAMETERS:",
    sum(
        p.numel()
        for p in encoder.parameters()
        if p.requires_grad
    )
)

tokenizer_config.json:   0%|          | 0.00/282 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/546 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/905k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/423M [00:00<?, ?B/s]

HIDDEN_SIZE: 768
TRAINABLE ENCODER PARAMETERS: 0


In [16]:
# =========================================================
# PRECOMPUTE FROZEN ENCODER OUTPUTS ONCE
# =========================================================

class RawEmotionDataset(Dataset):

    def __init__(
        self,
        frame
    ):

        self.texts = (
            frame[TEXT_COLUMN]
            .tolist()
        )

        self.labels = (
            frame[LABEL_COLUMNS]
            .values
            .astype(np.float32)
        )

    def __len__(self):
        return len(self.texts)

    def __getitem__(
        self,
        idx
    ):

        text = str(
            self.texts[idx]
        )

        encoding = tokenizer(
            text,
            truncation=True,
            padding="max_length",
            max_length=MAX_LEN,
            return_tensors="pt"
        )

        return {
            "input_ids":
                encoding["input_ids"]
                .squeeze(0),

            "attention_mask":
                encoding["attention_mask"]
                .squeeze(0),

            "labels":
                torch.tensor(
                    self.labels[idx],
                    dtype=torch.float32
                )
        }


@torch.no_grad()
def precompute_embeddings(
    frame,
    split_name
):

    loader = DataLoader(
        RawEmotionDataset(frame),
        batch_size=CACHE_BATCH_SIZE,
        shuffle=False,
        pin_memory=torch.cuda.is_available()
    )

    all_hidden = []
    all_masks = []
    all_labels = []

    use_amp = (
        PRIMARY_DEVICE.type == "cuda"
    )

    for batch in tqdm(
        loader,
        desc=f"Encoding {split_name}"
    ):

        input_ids = (
            batch["input_ids"]
            .to(
                PRIMARY_DEVICE,
                non_blocking=True
            )
        )

        attention_mask = (
            batch["attention_mask"]
            .to(
                PRIMARY_DEVICE,
                non_blocking=True
            )
        )

        with torch.cuda.amp.autocast(
            enabled=use_amp
        ):

            hidden = encoder(
                input_ids=input_ids,
                attention_mask=attention_mask
            ).last_hidden_state

        all_hidden.append(
            hidden
            .half()
            .cpu()
        )

        all_masks.append(
            attention_mask
            .bool()
            .cpu()
        )

        all_labels.append(
            batch["labels"]
            .cpu()
        )

    return (
        torch.cat(all_hidden, dim=0),
        torch.cat(all_masks, dim=0),
        torch.cat(all_labels, dim=0)
    )


print("Encoding TRAIN...")
train_hidden, train_masks, train_labels = precompute_embeddings(
    train_df,
    "train"
)

print("Encoding VAL...")
val_hidden, val_masks, val_labels = precompute_embeddings(
    val_df,
    "validation"
)

print("Encoding TEST...")
test_hidden, test_masks, test_labels = precompute_embeddings(
    test_df,
    "test"
)

print("TRAIN CACHE:", train_hidden.shape)
print("VAL CACHE  :", val_hidden.shape)
print("TEST CACHE :", test_hidden.shape)

# encoder ไม่ต้องอยู่บน GPU อีกแล้ว
del encoder

gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()

Encoding TRAIN...


Encoding train:   0%|          | 0/54 [00:01<?, ?it/s]

Encoding VAL...


Encoding validation:   0%|          | 0/6 [00:00<?, ?it/s]

Encoding TEST...


Encoding test:   0%|          | 0/15 [00:00<?, ?it/s]

TRAIN CACHE: torch.Size([1728, 128, 768])
VAL CACHE  : torch.Size([192, 128, 768])
TEST CACHE : torch.Size([480, 128, 768])


In [17]:
# =========================================================
# CACHED DATASET + SEEDED DATALOADER
# =========================================================

class CachedDataset(Dataset):

    def __init__(
        self,
        hidden,
        masks,
        labels
    ):
        self.hidden = hidden
        self.masks = masks
        self.labels = labels

    def __len__(self):
        return len(self.hidden)

    def __getitem__(
        self,
        idx
    ):
        return {
            "hidden": self.hidden[idx],
            "attention_mask": self.masks[idx],
            "labels": self.labels[idx]
        }


train_dataset = CachedDataset(
    train_hidden,
    train_masks,
    train_labels
)

val_dataset = CachedDataset(
    val_hidden,
    val_masks,
    val_labels
)

test_dataset = CachedDataset(
    test_hidden,
    test_masks,
    test_labels
)


def make_loaders(seed):

    generator = torch.Generator()

    generator.manual_seed(
        seed
    )

    common = dict(
        batch_size=BATCH_SIZE,
        num_workers=0,
        pin_memory=torch.cuda.is_available()
    )

    train_loader = DataLoader(
        train_dataset,
        shuffle=True,
        generator=generator,
        **common
    )

    val_loader = DataLoader(
        val_dataset,
        shuffle=False,
        **common
    )

    test_loader = DataLoader(
        test_dataset,
        shuffle=False,
        **common
    )

    return (
        train_loader,
        val_loader,
        test_loader
    )

In [18]:
# =========================================================
# ATTENTION POOLING — SAME AS ORIGINAL
# =========================================================

class AttentionPooling(nn.Module):

    def __init__(
        self,
        hidden_size
    ):

        super().__init__()

        self.attention = nn.Sequential(
            nn.Linear(
                hidden_size,
                hidden_size
            ),
            nn.Tanh(),
            nn.Linear(
                hidden_size,
                1
            )
        )

    def forward(
        self,
        x,
        mask
    ):

        scores = (
            self.attention(x)
            .squeeze(-1)
        )

        scores = scores.masked_fill(
            mask == 0,
            torch.finfo(
                scores.dtype
            ).min
        )

        weights = torch.softmax(
            scores,
            dim=1
        )

        pooled = torch.sum(
            x
            * weights.unsqueeze(-1),
            dim=1
        )

        return pooled

In [19]:
# =========================================================
# DECODER MODEL — SAME STRUCTURE, ENCODER REMOVED/FROZEN
# =========================================================

class EmotionModel(nn.Module):

    def __init__(self):

        super().__init__()

        p = FIXED_BEST_PARAMS

        self.decoder = Decoder(
            dim=HIDDEN_SIZE,
            depth=p["depth"],
            heads=p["heads"],
            attn_dropout=p["attn_dropout"],
            ff_dropout=p["ff_dropout"],
            ff_mult=p["ff_mult"]
        )

        self.pooling = AttentionPooling(
            HIDDEN_SIZE
        )

        # ต้นฉบับ = 0.4
        self.dropout = nn.Dropout(
            0.4
        )

        self.fc = nn.Linear(
            HIDDEN_SIZE,
            NUM_LABELS
        )

    def forward(
        self,
        hidden,
        attention_mask
    ):

        # DataParallel จะ scatter tensor ไป GPU แต่ละใบเอง
        # จึงห้าม force กลับ cuda:0 ภายใน forward
        x = hidden.float()
        mask = attention_mask

        # ต้นฉบับเรียก Decoder(x) โดยไม่ส่ง mask
        x = self.decoder(
            x
        )

        x = self.pooling(
            x,
            mask
        )

        x = self.dropout(
            x
        )

        logits = self.fc(
            x
        )

        return logits

In [20]:
# =========================================================
# THRESHOLD TUNING — EXACT ORIGINAL GRID
# =========================================================

def optimize_thresholds(
    y_true,
    y_probs
):

    best_thresholds = []

    for i in range(
        NUM_LABELS
    ):

        best_thr = 0.5
        best_f1 = 0.0

        for thr in np.arange(
            0.1,
            0.9,
            0.05
        ):

            preds = (
                y_probs[:, i]
                >= thr
            ).astype(int)

            score = f1_score(
                y_true[:, i],
                preds,
                zero_division=0
            )

            if score > best_f1:

                best_f1 = score
                best_thr = thr

        best_thresholds.append(
            best_thr
        )

    return np.array(
        best_thresholds
    )

In [21]:
# =========================================================
# EVALUATION
# =========================================================

@torch.no_grad()
def collect_probs(
    model,
    loader
):

    model.eval()

    all_labels = []
    all_probs = []

    for batch in loader:

        hidden = batch["hidden"].to(
            PRIMARY_DEVICE,
            non_blocking=True
        )

        mask = batch["attention_mask"].to(
            PRIMARY_DEVICE,
            non_blocking=True
        )

        logits = model(
            hidden,
            mask
        )

        probs = torch.sigmoid(
            logits
        )

        all_probs.append(
            probs
            .cpu()
            .numpy()
        )

        all_labels.append(
            batch["labels"]
            .numpy()
        )

    return (
        np.vstack(all_labels),
        np.vstack(all_probs)
    )


def calculate_metrics(
    y_true,
    y_probs,
    thresholds
):

    preds = (
        y_probs
        >= thresholds
    ).astype(int)

    per_precision = precision_score(
        y_true,
        preds,
        average=None,
        zero_division=0
    )

    per_recall = recall_score(
        y_true,
        preds,
        average=None,
        zero_division=0
    )

    per_f1 = f1_score(
        y_true,
        preds,
        average=None,
        zero_division=0
    )

    return {
        "accuracy":
            accuracy_score(
                y_true,
                preds
            ),

        "micro_precision":
            precision_score(
                y_true,
                preds,
                average="micro",
                zero_division=0
            ),

        "micro_recall":
            recall_score(
                y_true,
                preds,
                average="micro",
                zero_division=0
            ),

        "micro_f1":
            f1_score(
                y_true,
                preds,
                average="micro",
                zero_division=0
            ),

        "macro_precision":
            precision_score(
                y_true,
                preds,
                average="macro",
                zero_division=0
            ),

        "macro_recall":
            recall_score(
                y_true,
                preds,
                average="macro",
                zero_division=0
            ),

        "macro_f1":
            f1_score(
                y_true,
                preds,
                average="macro",
                zero_division=0
            ),

        "weighted_f1":
            f1_score(
                y_true,
                preds,
                average="weighted",
                zero_division=0
            ),

        "hamming_loss":
            hamming_loss(
                y_true,
                preds
            ),

        "per_precision":
            per_precision,

        "per_recall":
            per_recall,

        "per_f1":
            per_f1,

        "support":
            y_true
            .sum(axis=0)
            .astype(int),

        "preds":
            preds
    }

In [22]:
# =========================================================
# TRAIN ONE SEED — 100 EPOCHS
# Same optimizer/loss/scheduler logic as original
# =========================================================

def train_one_seed(seed):

    seed_everything(
        seed
    )

    (
        train_loader,
        val_loader,
        test_loader
    ) = make_loaders(
        seed
    )

    model = EmotionModel().to(
        PRIMARY_DEVICE
    )

    if USE_DATA_PARALLEL:
        model = nn.DataParallel(
            model,
            device_ids=list(range(GPU_COUNT))
        )
        print("DataParallel GPUs:", list(range(GPU_COUNT)))

    pos_weights = torch.tensor(
        pos_weights_np,
        dtype=torch.float32,
        device=PRIMARY_DEVICE
    )

    criterion = nn.BCEWithLogitsLoss(
        pos_weight=pos_weights
    )

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=FIXED_BEST_PARAMS["lr"],
        weight_decay=FIXED_BEST_PARAMS["weight_decay"]
    )

    total_steps = (
        len(train_loader)
        * EPOCHS
    )

    scheduler = get_linear_schedule_with_warmup(
        optimizer,
        num_warmup_steps=int(
            0.1
            * total_steps
        ),
        num_training_steps=total_steps
    )

    use_amp = (
        PRIMARY_DEVICE.type == "cuda"
    )

    scaler = torch.cuda.amp.GradScaler(
        enabled=use_amp
    )

    start_time = time.perf_counter()

    for epoch in range(
        EPOCHS
    ):

        model.train()

        total_loss = 0.0

        for batch in train_loader:

            labels = (
                batch["labels"]
                .to(
                    PRIMARY_DEVICE
                )
            )

            optimizer.zero_grad(
                set_to_none=True
            )

            with torch.cuda.amp.autocast(
                enabled=use_amp
            ):

                hidden = batch["hidden"].to(
                    PRIMARY_DEVICE,
                    non_blocking=True
                )

                mask = batch["attention_mask"].to(
                    PRIMARY_DEVICE,
                    non_blocking=True
                )

                logits = model(
                    hidden,
                    mask
                )

                loss = criterion(
                    logits,
                    labels
                )

            scaler.scale(
                loss
            ).backward()

            scaler.unscale_(
                optimizer
            )

            torch.nn.utils.clip_grad_norm_(
                model.parameters(),
                1.0
            )

            scaler.step(
                optimizer
            )

            scaler.update()

            scheduler.step()

            total_loss += (
                loss.item()
            )

        # display validation at current epoch using 0.5 threshold,
        # matching the original training display
        val_y, val_prob = collect_probs(
            model,
            val_loader
        )

        default_thresholds = np.array(
            [0.5] * NUM_LABELS
        )

        val_metrics = calculate_metrics(
            val_y,
            val_prob,
            default_thresholds
        )

        print(
            f"seed={seed} | "
            f"epoch {epoch+1:03d}/{EPOCHS} | "
            f"loss={total_loss/len(train_loader):.4f} | "
            f"val_acc={val_metrics['accuracy']:.4f} | "
            f"val_micro={val_metrics['micro_f1']:.4f} | "
            f"val_macro={val_metrics['macro_f1']:.4f}"
        )

    # ใช้ final epoch model ตาม notebook เดิม
    # tune thresholds บน validation หลัง train เสร็จ
    val_y, val_prob = collect_probs(
        model,
        val_loader
    )

    best_thresholds = optimize_thresholds(
        val_y,
        val_prob
    )

    test_y, test_prob = collect_probs(
        model,
        test_loader
    )

    test_metrics = calculate_metrics(
        test_y,
        test_prob,
        best_thresholds
    )

    elapsed = (
        time.perf_counter()
        - start_time
    )

    del model
    del optimizer
    del scheduler

    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return (
        test_metrics,
        best_thresholds,
        elapsed
    )

In [23]:
# =========================================================
# RUN 5 INDEPENDENT SEEDS
# =========================================================

overall_rows = []
per_label_rows = []

for run_idx, seed in enumerate(
    TRAIN_SEEDS,
    start=1
):

    print(
        "\n"
        + "=" * 90
    )

    print(
        f"RUN {run_idx}/5 | seed={seed}"
    )

    print(
        "=" * 90
    )

    (
        result,
        thresholds,
        elapsed
    ) = train_one_seed(
        seed
    )

    row = {
        "run": run_idx,
        "seed": seed,

        "accuracy":
            result["accuracy"],

        "micro_precision":
            result["micro_precision"],

        "micro_recall":
            result["micro_recall"],

        "micro_f1":
            result["micro_f1"],

        "macro_precision":
            result["macro_precision"],

        "macro_recall":
            result["macro_recall"],

        "macro_f1":
            result["macro_f1"],

        "weighted_f1":
            result["weighted_f1"],

        "hamming_loss":
            result["hamming_loss"],

        "minutes":
            elapsed / 60.0
    }

    overall_rows.append(
        row
    )

    print("\nTEST RESULT")

    for key, value in row.items():

        if key not in [
            "run",
            "seed"
        ]:

            print(
                f"{key:20s}: "
                f"{value:.6f}"
            )

    print("\nTHRESHOLDS")

    for label, thr in zip(
        LABEL_COLUMNS,
        thresholds
    ):

        print(
            f"{label:12s}: "
            f"{thr:.2f}"
        )

    for i, label in enumerate(
        LABEL_COLUMNS
    ):

        per_label_rows.append({
            "run":
                run_idx,

            "seed":
                seed,

            "label":
                label,

            "precision":
                result[
                    "per_precision"
                ][i],

            "recall":
                result[
                    "per_recall"
                ][i],

            "f1":
                result[
                    "per_f1"
                ][i],

            "support":
                int(
                    result[
                        "support"
                    ][i]
                ),

            "threshold":
                float(
                    thresholds[i]
                )
        })


    # autosave หลังจบแต่ละ run เผื่อ Kaggle session หลุด
    pd.DataFrame(overall_rows).to_csv(
        os.path.join(OUTPUT_DIR, "sied_original_5seeds_100epochs_runs_PARTIAL.csv"),
        index=False,
        encoding="utf-8-sig"
    )

    pd.DataFrame(per_label_rows).to_csv(
        os.path.join(OUTPUT_DIR, "sied_original_5seeds_100epochs_per_label_PARTIAL.csv"),
        index=False,
        encoding="utf-8-sig"
    )


results_df = pd.DataFrame(
    overall_rows
)

per_label_df = pd.DataFrame(
    per_label_rows
)

print("\nALL 5 RUNS")
display(results_df)

print("\nPER-LABEL RAW RESULTS")
display(per_label_df)


RUN 1/5 | seed=13
DataParallel GPUs: [0, 1]
seed=13 | epoch 001/100 | loss=1.2489 | val_acc=0.0000 | val_micro=0.4409 | val_macro=0.3022
seed=13 | epoch 002/100 | loss=1.1841 | val_acc=0.0156 | val_micro=0.4644 | val_macro=0.3516
seed=13 | epoch 003/100 | loss=1.1317 | val_acc=0.0521 | val_micro=0.5180 | val_macro=0.3923
seed=13 | epoch 004/100 | loss=1.0955 | val_acc=0.0625 | val_micro=0.5205 | val_macro=0.4029
seed=13 | epoch 005/100 | loss=1.0321 | val_acc=0.2031 | val_micro=0.6073 | val_macro=0.4584
seed=13 | epoch 006/100 | loss=0.9679 | val_acc=0.2500 | val_micro=0.6341 | val_macro=0.4873
seed=13 | epoch 007/100 | loss=0.8829 | val_acc=0.2552 | val_micro=0.6366 | val_macro=0.4911
seed=13 | epoch 008/100 | loss=0.8113 | val_acc=0.2656 | val_micro=0.6447 | val_macro=0.5009
seed=13 | epoch 009/100 | loss=0.7429 | val_acc=0.2396 | val_micro=0.6350 | val_macro=0.5183
seed=13 | epoch 010/100 | loss=0.6934 | val_acc=0.3177 | val_micro=0.6851 | val_macro=0.5263
seed=13 | epoch 011/100 |

,run,seed,accuracy,micro_precision,micro_recall,micro_f1,macro_precision,macro_recall,macro_f1,weighted_f1,hamming_loss,minutes
0,1,13,0.502083,0.717095,0.741784,0.729231,0.542088,0.483737,0.500671,0.718754,0.122222,1.924177
1,2,21,0.460417,0.692194,0.735524,0.713202,0.580299,0.497528,0.497787,0.717429,0.131250,1.915965
2,3,42,0.456250,0.685083,0.776213,0.727806,0.526419,0.532582,0.516191,0.731733,0.128819,1.912466
3,4,87,0.481250,0.704204,0.733959,0.718774,0.522219,0.494098,0.499411,0.713117,0.127431,1.913815
4,5,123,0.483333,0.694964,0.755869,0.724138,0.537828,0.500275,0.504010,0.722947,0.127778,1.910772



PER-LABEL RAW RESULTS


,run,seed,label,precision,recall,f1,support,threshold
0,1,13,Happiness,0.727273,0.457143,0.561404,35,0.75
1,1,13,Sadness,0.875294,0.961240,0.916256,387,0.10
2,1,13,Anger,0.411765,0.264151,0.321839,53,0.80
3,1,13,Disgust,0.381356,0.529412,0.443350,85,0.10
4,1,13,Surprise,0.384615,0.357143,0.370370,28,0.40
5,1,13,Fear,0.472222,0.333333,0.390805,51,0.80
6,2,21,Happiness,0.636364,0.400000,0.491228,35,0.75
7,2,21,Sadness,0.896465,0.917313,0.906769,387,0.55
8,2,21,Anger,0.338235,0.433962,0.380165,53,0.10
9,2,21,Disgust,0.409836,0.588235,0.483092,85,0.10


In [24]:
# =========================================================
# MEAN / MEDIAN / VARIANCE / STANDARD DEVIATION
# =========================================================

metric_columns = [
    "accuracy",
    "micro_precision",
    "micro_recall",
    "micro_f1",
    "macro_precision",
    "macro_recall",
    "macro_f1",
    "weighted_f1",
    "hamming_loss"
]

summary_rows = []

for metric in metric_columns:

    values = (
        results_df[metric]
        .astype(float)
    )

    summary_rows.append({
        "metric":
            metric,

        "mean":
            values.mean(),

        "median":
            values.median(),

        # sample variance
        "variance":
            values.var(
                ddof=1
            ),

        # sample standard deviation
        "std":
            values.std(
                ddof=1
            ),

        "min":
            values.min(),

        "max":
            values.max()
    })


summary_df = pd.DataFrame(
    summary_rows
)

print(
    "5-RUN STATISTICAL SUMMARY"
)

display(
    summary_df
)

5-RUN STATISTICAL SUMMARY


,metric,mean,median,variance,std,min,max
0,accuracy,0.476667,0.481250,0.000348,0.018657,0.456250,0.502083
1,micro_precision,0.698708,0.694964,0.000153,0.012353,0.685083,0.717095
2,micro_recall,0.748670,0.741784,0.000312,0.017657,0.733959,0.776213
3,micro_f1,0.722630,0.724138,0.000044,0.006641,0.713202,0.729231
4,macro_precision,0.541770,0.537828,0.000530,0.023011,0.522219,0.580299
5,macro_recall,0.501644,0.497528,0.000338,0.018396,0.483737,0.532582
6,macro_f1,0.503614,0.500671,0.000055,0.007393,0.497787,0.516191
7,weighted_f1,0.720796,0.718754,0.000050,0.007048,0.713117,0.731733
8,hamming_loss,0.127500,0.127778,0.000011,0.003307,0.122222,0.131250


In [25]:
# =========================================================
# PER-LABEL SUMMARY ACROSS 5 RUNS
# =========================================================

per_label_summary = (
    per_label_df
    .groupby(
        "label",
        sort=False
    )
    .agg(
        precision_mean=(
            "precision",
            "mean"
        ),

        precision_median=(
            "precision",
            "median"
        ),

        precision_variance=(
            "precision",
            lambda x: x.var(
                ddof=1
            )
        ),

        precision_std=(
            "precision",
            lambda x: x.std(
                ddof=1
            )
        ),

        recall_mean=(
            "recall",
            "mean"
        ),

        recall_median=(
            "recall",
            "median"
        ),

        recall_variance=(
            "recall",
            lambda x: x.var(
                ddof=1
            )
        ),

        recall_std=(
            "recall",
            lambda x: x.std(
                ddof=1
            )
        ),

        f1_mean=(
            "f1",
            "mean"
        ),

        f1_median=(
            "f1",
            "median"
        ),

        f1_variance=(
            "f1",
            lambda x: x.var(
                ddof=1
            )
        ),

        f1_std=(
            "f1",
            lambda x: x.std(
                ddof=1
            )
        )
    )
    .reset_index()
)

print(
    "PER-LABEL 5-RUN SUMMARY"
)

display(
    per_label_summary
)

PER-LABEL 5-RUN SUMMARY


,label,precision_mean,precision_median,precision_variance,precision_std,recall_mean,recall_median,recall_variance,recall_std,f1_mean,f1_median,f1_variance,f1_std
0,Happiness,0.660918,0.666667,0.004518,0.067215,0.428571,0.428571,0.000816,0.028571,0.519477,0.535714,0.001533,0.039156
1,Sadness,0.880721,0.878220,0.000088,0.009397,0.955556,0.961240,0.000485,0.022032,0.916407,0.918317,0.000032,0.005695
2,Anger,0.374074,0.370370,0.001836,0.042846,0.362264,0.377358,0.004877,0.069837,0.363739,0.373832,0.001695,0.041172
3,Disgust,0.411664,0.417910,0.000345,0.018565,0.515294,0.541176,0.011516,0.107311,0.452834,0.471795,0.002622,0.051202
4,Surprise,0.582637,0.600000,0.035558,0.188567,0.328571,0.321429,0.005995,0.077427,0.401829,0.370370,0.004002,0.063263
5,Fear,0.340609,0.303797,0.005874,0.076642,0.419608,0.431373,0.003383,0.058166,0.367397,0.369231,0.000461,0.021473


In [26]:
# =========================================================
# PAPER-READY TABLE
# Mean ± SD for each emotion
# =========================================================

paper_per_label = per_label_summary[
    [
        "label",
        "precision_mean",
        "precision_std",
        "recall_mean",
        "recall_std",
        "f1_mean",
        "f1_std"
    ]
].copy()

paper_per_label["Precision (Mean±SD)"] = (
    paper_per_label["precision_mean"].map(
        lambda x: f"{x:.4f}"
    )
    + " ± "
    + paper_per_label["precision_std"].map(
        lambda x: f"{x:.4f}"
    )
)

paper_per_label["Recall (Mean±SD)"] = (
    paper_per_label["recall_mean"].map(
        lambda x: f"{x:.4f}"
    )
    + " ± "
    + paper_per_label["recall_std"].map(
        lambda x: f"{x:.4f}"
    )
)

paper_per_label["F1-score (Mean±SD)"] = (
    paper_per_label["f1_mean"].map(
        lambda x: f"{x:.4f}"
    )
    + " ± "
    + paper_per_label["f1_std"].map(
        lambda x: f"{x:.4f}"
    )
)

paper_per_label = paper_per_label[
    [
        "label",
        "Precision (Mean±SD)",
        "Recall (Mean±SD)",
        "F1-score (Mean±SD)"
    ]
].rename(
    columns={
        "label": "Emotion"
    }
)

display(
    paper_per_label
)

,Emotion,Precision (Mean±SD),Recall (Mean±SD),F1-score (Mean±SD)
0,Happiness,0.6609 ± 0.0672,0.4286 ± 0.0286,0.5195 ± 0.0392
1,Sadness,0.8807 ± 0.0094,0.9556 ± 0.0220,0.9164 ± 0.0057
2,Anger,0.3741 ± 0.0428,0.3623 ± 0.0698,0.3637 ± 0.0412
3,Disgust,0.4117 ± 0.0186,0.5153 ± 0.1073,0.4528 ± 0.0512
4,Surprise,0.5826 ± 0.1886,0.3286 ± 0.0774,0.4018 ± 0.0633
5,Fear,0.3406 ± 0.0766,0.4196 ± 0.0582,0.3674 ± 0.0215


In [27]:
# =========================================================
# EXPORT CSV
# =========================================================

paths = {
    "runs":
        os.path.join(
            OUTPUT_DIR,
            "sied_original_5seeds_100epochs_runs.csv"
        ),

    "summary":
        os.path.join(
            OUTPUT_DIR,
            "sied_original_5seeds_100epochs_summary.csv"
        ),

    "per_label_raw":
        os.path.join(
            OUTPUT_DIR,
            "sied_original_5seeds_100epochs_per_label_raw.csv"
        ),

    "per_label_summary":
        os.path.join(
            OUTPUT_DIR,
            "sied_original_5seeds_100epochs_per_label_summary.csv"
        ),

    "paper_table":
        os.path.join(
            OUTPUT_DIR,
            "sied_original_5seeds_100epochs_paper_table.csv"
        )
}


results_df.to_csv(
    paths["runs"],
    index=False,
    encoding="utf-8-sig"
)

summary_df.to_csv(
    paths["summary"],
    index=False,
    encoding="utf-8-sig"
)

per_label_df.to_csv(
    paths["per_label_raw"],
    index=False,
    encoding="utf-8-sig"
)

per_label_summary.to_csv(
    paths["per_label_summary"],
    index=False,
    encoding="utf-8-sig"
)

paper_per_label.to_csv(
    paths["paper_table"],
    index=False,
    encoding="utf-8-sig"
)


print("EXPORTED FILES")

for name, path in paths.items():

    print(
        name,
        "->",
        path
    )

EXPORTED FILES
runs -> /kaggle/working/sied_original_5seeds_100epochs_runs.csv
summary -> /kaggle/working/sied_original_5seeds_100epochs_summary.csv
per_label_raw -> /kaggle/working/sied_original_5seeds_100epochs_per_label_raw.csv
per_label_summary -> /kaggle/working/sied_original_5seeds_100epochs_per_label_summary.csv
paper_table -> /kaggle/working/sied_original_5seeds_100epochs_paper_table.csv
